#SIS4309 - Procesamiento Inteligente de Datos

### SIS4309_Formato-Python

* Ejercicio Unidad 4
* 4. Redes neuronales artificiales
* SIS4309_Python_MirandaHau_Tarea-4

**Datos del estudiante:**
* Nombre completo:
* Mátricula:


Profesor: Enrique Naredo García

#Introducción

- En esta práctica se desarrolla un modelo de **clasificación binaria** para predecir la aparición de diabetes (0 = no, 1 = sí) usando redes neuronales con **Keras (TensorFlow)**.
- Se utiliza el dataset `pima-indians-diabetes.csv` (UCI), que contiene 8 variables de entrada y 1 variable objetivo.
- Objetivos:
  - Cargar y explorar el conjunto de datos.
  - Preprocesar (escalado y separación entrenamiento/prueba).
  - Definir, compilar y entrenar un modelo denso (MLP).
  - Evaluar rendimiento con métricas y visualizaciones.
- Notas:
  - Se usan nombres de variables en español y comentarios en español para facilitar la lectura.


#Importar librerías

In [ ]:
# Librerías básicas
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt

# Preprocesamiento y métricas
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Para mostrar más filas/columnas si es necesario
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)

# Semilla para reproducibilidad (no garantiza resultados idénticos en todos los entornos)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

#Subir y cargar *datos*

In [ ]:
# Se ejecuta esta celda y se usa el botón para seleccionar el archivo pima-indians-diabetes.csv
from google.colab import files
uploaded = files.upload()  # selecciona pima-indians-diabetes.csv desde tu equipo

# Al usar files.upload() el diccionario 'uploaded' contendrá el archivo subido.
nombre_archivo = list(uploaded.keys())[0]
print("Archivo subido:", nombre_archivo)

##Exploración inicial y renombrar columnas (celda de código)

In [ ]:
# Asignar nombres de columnas en español (8 entradas + 1 salida)
nombres = [
    'embarazos', 'glucosa', 'presion_diastolica', 'pliegue_tricipital',
    'insulina_2h', 'imc', 'funcion_diabetes', 'edad', 'resultado'
]
df.columns = nombres

# Mostrar las primeras 10 filas y estadísticas básicas
display(df.head(10))
display(df.describe().round(3))

#Limpieza y preprocesamiento

In [ ]:
# Columnas donde 0 no tiene sentido clínico y se tratarán como NaN:
columnas_cero_nan = ['glucosa', 'presion_diastolica', 'pliegue_tricipital', 'insulina_2h', 'imc']

df_limpio = df.copy()
# Reemplazamos ceros por NaN en las columnas seleccionadas
df_limpio[columnas_cero_nan] = df_limpio[columnas_cero_nan].replace(0, np.nan)

# Imputación simple: mediana (robusta frente a outliers)
for col in columnas_cero_nan:
    mediana = df_limpio[col].median()
    df_limpio[col].fillna(mediana, inplace=True)

# Verificamos que no queden NaN
print("Valores faltantes por columna después de imputar:")
print(df_limpio.isnull().sum())

# Separar X e y
X = df_limpio.drop('resultado', axis=1).values
y = df_limpio['resultado'].values

# División entrenamiento / prueba
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Escalado de características (StandardScaler)
escalador = StandardScaler()
X_entrena_escalado = escalador.fit_transform(X_entrena)
X_prueba_escalado = escalador.transform(X_prueba)

print("Tamaño entrenamiento:", X_entrena_escalado.shape)
print("Tamaño prueba:", X_prueba_escalado.shape)

#Definición del modelo

In [ ]:
# Definir el modelo con nombres de variables en español
modelo = Sequential(name='MLP_Diabetes')

# Primera capa oculta (definimos input_shape aquí)
modelo.add(Dense(14, input_shape=(X_entrena_escalado.shape[1],), activation='relu', name='capa_oculta_1'))
modelo.add(Dense(8, activation='relu', name='capa_oculta_2'))
modelo.add(Dense(1, activation='sigmoid', name='capa_salida'))

# Compilar el modelo
modelo.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Resumen del modelo
modelo.summary()

#Entrenamiento

In [ ]:
# Callbacks: EarlyStopping para detener si no hay mejora en val_loss
callback_parada = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Entrenar el modelo
historial = modelo.fit(
    X_entrena_escalado, y_entrena,
    validation_split=0.2,  # dividir parte del entrenamiento para validación
    epochs=200,
    batch_size=16,
    callbacks=[callback_parada],
    verbose=1
)

#Visualizar curvas de entrenamiento

In [ ]:
# Gráficas de pérdida y precisión
plt.figure(figsize=(12,5))

# Pérdida
plt.subplot(1,2,1)
plt.plot(historial.history['loss'], label='loss entreno')
plt.plot(historial.history['val_loss'], label='loss validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Precisión
plt.subplot(1,2,2)
plt.plot(historial.history['accuracy'], label='accuracy entreno')
plt.plot(historial.history['val_accuracy'], label='accuracy validación')
plt.title('Precisión (Accuracy)')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

#Evaluación en el conjunto de prueba

In [ ]:
# Evaluar en datos de prueba
loss_prueba, acc_prueba = modelo.evaluate(X_prueba_escalado, y_prueba, verbose=0)
print(f"Loss prueba: {loss_prueba:.4f}")
print(f"Accuracy prueba: {acc_prueba*100:.2f}%")

# Predicciones
pred_proba = modelo.predict(X_prueba_escalado)
pred_clase = (pred_proba > 0.5).astype(int).flatten()

# Matriz de confusión y reporte de clasificación
cm = confusion_matrix(y_prueba, pred_clase)
print("\nMatriz de confusión:\n", cm)
print("\nReporte de clasificación:\n", classification_report(y_prueba, pred_clase, digits=4))

#Curva ROC

In [ ]:
# Curva ROC y AUC
fpr, tpr, umbrales = roc_curve(y_prueba, pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0,1], [0,1], linestyle='--', label='Azar')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.show()

#Mostrar ejemplos de predicción

In [ ]:
# Mostrar primeros 10 ejemplos de prueba con predicción y clase real
print("Ejemplos de prueba - entrada -> predicción (prob) -> predicción (clase) -> real\n")
for i in range(10):
    entrada = X_prueba[i].tolist()
    print(f"{[round(x,3) for x in entrada]} -> {pred_proba[i][0]:.3f} -> {pred_clase[i]} -> {y_prueba[i]}")

#Guardar modelo

In [ ]:
# Guardar modelo en archivo .h5 (opcional)
modelo.save('modelo_mlp_diabetes.h5')
print("Modelo guardado como modelo_mlp_diabetes.h5")

# Conclusiones

En este trabajo se logró entrenar una red neuronal sencilla (MLP) para predecir si una persona puede desarrollar diabetes usando el dataset Pima Indians. Usar Keras hizo que la implementación fuera práctica y clara, y con herramientas como sklearn pudimos preparar los datos fácilmente. También ayudó mucho usar EarlyStopping para evitar que el modelo se sobreentrenara y el escalado de datos (StandardScaler) mejoró el aprendizaje.

Sin embargo, el conjunto de datos es pequeño, por lo que los resultados pueden variar entre una prueba y otra. Además, la forma en que se completaron los datos faltantes fue simple y no refleja completamente la realidad. Tampoco se hizo una prueba profunda de otros parámetros del modelo, como capas, neuronas o tipos de optimizadores, lo cual podría mejorar el rendimiento.

Como mejora, sería buena idea probar validación cruzada (K-fold), ajustar hiperparámetros de forma más precisa y comparar este modelo con otros más simples, como regresión logística o Random Forest, para tener una referencia clara.


##Bibiliografía

- Brownlee, J. (2017). *How to create your first neural network in Python with Keras*. Machine Learning Mastery. Recuperado de https://machinelearningmastery.com/tutorial-first-neural-network-python-keras/  

- UCI Machine Learning Repository. (s.f.). *Pima Indians Diabetes Database*. Recuperado de https://archive.ics.uci.edu/ml/datasets/pima+indians+diabetes  

- Chollet, F., & others. (2015). *Keras documentation*. https://keras.io

- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Vanderplas, J. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825-2830.